
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [2]:
from langchain_groq import ChatGroq

model=ChatGroq(model='openai/gpt-oss-20b')

In [3]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001959A2DAD10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001959A43E810>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year the movie was released")
    genre:str=Field(...,description="The genre of the movie")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie")

In [5]:
model_with_str=model.with_structured_output(Movie)
model_with_str

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001959A2DAD10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001959A43E810>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The tit

In [6]:
model.invoke("Provide details about the moview Inception")

AIMessage(content="**Inception**  \n*Released:* 2010 (July 16, USA)  \n*Director:* Christopher\u202fNolan  \n*Genre:* Sci‑fi Thriller / Action  \n*Runtime:* 2\u202fh\u202f28\u202fmin  \n*Budget:* $160\u202fM (approx.)  \n*Box Office:* $828\u202fM worldwide  \n\n---\n\n## 1. Core Crew & Cast\n\n| Role | Person |\n|------|--------|\n| Director | Christopher\u202fNolan |\n| Writer | Christopher\u202fNolan |\n| Producer | Emma Thomas, Christopher\u202fNolan, Emma Thomas |\n| Cinematographer | Wally\u202fPfister |\n| Editor | Lee\u202fSmith |\n| Composer | Hans\u202fZimmer |\n| Production Designer | Guy\u202fRitchie |\n| **Main Cast** | |\n| Dom\u202fCobb | Leonardo\u202fDiCaprio |\n| Arthur | Joseph\u202fGordon‑Levitt |\n| Ariadne | Ellen\u202fPage |\n| Eames | Tom\u202fHardy |\n| Yusuf | Dileep\u202fNair |\n| Mal (Cobb’s wife) | Marion\u202fCotillard |\n| Robert\u202fSaito | Ken\u202fWatanabe |\n| Saito’s wife | Ken\u202fWatanabe (cameo) |\n\n---\n\n## 2. Plot (Spoiler‑Alert)\n\nDom\u202f

In [7]:
response=model_with_str.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan', rating=8.8)

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.


In [9]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [10]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [11]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [12]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent

In [13]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='f59eb416-325d-4ea3-a0c3-fc98e6d385fd'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON object matching schema. Must be compact JSON, parseable. Include all required fields: name, email, phone. Provide values: name "John Doe", email "john@example.com", phone "(555) 123-4567". Ensure no extra whitespace? compact: no line breaks, minimal spaces. Use double quotes. Provide JSON object only.'}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 238, 'total_tokens': 346, 'completion_time': 0.121339863, 'completion_tokens_details': {'reasoning_tokens': 76}, 'prompt_time': 0.015351759, 'prompt_tokens_details': None, 'queue_time': 0.314286947, 'total_time': 0.136691622}, 'model_name': 'openai/gpt-oss-20b'

In [14]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [15]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [16]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')